# HDBSCAN (Hierarchical Density-Based Spatial Clustering of Applications with Noise)

## Introduction

HDBSCAN is an **unsupervised**, **density-based**, and **hierarchical** clustering algorithm that extends DBSCAN.

It combines the advantages of:

- **Density-based clustering** (handles arbitrary shapes and varying densities)
- **Hierarchical clustering** (builds a hierarchy of nested clusters)

Unlike DBSCAN, HDBSCAN **does not require a fixed `eps` value** and can automatically determine the number of clusters.

---

## Clustering Taxonomy

Clustering algorithms can be classified in two ways.

### Based on Cluster Representation

1. **Centroid-Based (Parametric)**
2. **Density-Based (Non-Parametric)**

### Based on Output

1. **Flat Clustering** – Produces a single partition of the data.
2. **Hierarchical Clustering** – Produces a hierarchy of nested clusters at different levels.

|                     | Flat | Hierarchical |
|---------------------|------|--------------|
| **Centroid / Parametric** | K-Means, GMM | Ward, Complete Linkage |
| **Density / Non-Parametric** | DBSCAN, Mean Shift | HDBSCAN |

---

## Why HDBSCAN?

Since HDBSCAN is:

- **Density-based**, it naturally detects clusters with **varying densities** and arbitrary shapes.
- **Hierarchical**, it solves the **resolution problem** by generating clusters at multiple density levels instead of forcing a single clustering solution.

### Resolution Problem

The resolution problem is the inability of flat clustering algorithms to represent meaningful clusters that exist at different density or scale levels simultaneously.

---

## Cluster Tree

HDBSCAN constructs a **cluster tree**, where clusters appear, split, and disappear as the density threshold changes.

### Example

Suppose a dataset contains:

- Animals
  - Mammals
    - Dogs
    - Cats
  - Birds

The cluster tree first groups **Dogs** and **Cats** into **Mammals**, then merges **Mammals** and **Birds** into **Animals**.

Similarly, HDBSCAN builds nested clusters from dense local groups to larger clusters.

---

# Working of HDBSCAN

## Step 1: Approximate Local Density

Local density is estimated using one of two equivalent views:

- Draw a circle around each point containing at least `min_samples` neighbors.
- Alternatively, draw the **smallest possible circle** enclosing `min_samples` neighbors.

**Smaller radius ⇒ Higher local density**

The radius to the `k^{th}` nearest neighbor is called the **core distance**.

---

## Step 2: Compute Mutual Reachability Distance

Instead of Euclidean distance, HDBSCAN uses **Mutual Reachability Distance (MRD)**.

$$
\text{MRD}(a,b)=\max\left(\text{core}_k(a),\ \text{core}_k(b),\ d(a,b)\right)
$$

where:

- $\text{core}_k(x)$ = core distance of point $x$
- $d(a,b)$ = Euclidean distance

MRD increases distances through sparse regions while preserving dense connections.

---

## Step 3: Build the Cluster Tree

- Construct a **Minimum Spanning Tree (MST)** using the mutual reachability distances.
- As the density threshold decreases (or equivalently, allowed radius increases), nearby components merge.
- These successive merges form a **hierarchical cluster tree (dendrogram)**.

---

## Step 4: Condense the Tree

- Remove branches with fewer than `min_cluster_size` points.
- Small branches are treated as cluster shrinkage rather than separate clusters.

This produces the **condensed cluster tree**.

---

## Step 5: Extract Stable Clusters

Instead of choosing a fixed `eps` like DBSCAN, HDBSCAN selects clusters that remain **stable over the largest range of density levels**.

These stable clusters become the final output.

---

## Important Hyperparameters

### `min_cluster_size`

- Minimum number of points required to form a cluster.
- Controls the smallest cluster that can be detected.

### `min_samples`

- Number of neighbors used to estimate local density.
- Larger values produce more conservative clustering and detect more outliers.

### `metric`

- Distance metric used to compute distances.
- Common options include Euclidean, Manhattan, Cosine, and Minkowski.

---

## Time Complexity

### Naive Implementation

$$
O(n^2)
$$

### Optimized Implementation

Using **Borůvka's algorithm** with KD-Trees or Cover Trees:

$$
O(n\log n)
$$

---

## Advantages

- No need to specify the number of clusters.
- No fixed `eps` parameter.
- Detects clusters of varying densities.
- Handles arbitrary-shaped clusters.
- Robust to noise and outliers.
- Produces a hierarchical cluster structure.

---

## Limitations

- More computationally expensive than DBSCAN.
- More complex to understand and visualize.
- Performance depends on suitable values of `min_cluster_size` and `min_samples`.

---

## Comparison

| Algorithm | Need K? | Fixed `eps`? | Varying Density | Hierarchical | Detects Noise |
|-----------|:-------:|:------------:|:---------------:|:------------:|:-------------:|
| K-Means | ✓ | ✗ | ✗ | ✗ | ✗ |
| DBSCAN | ✗ | ✓ | ✗ | ✗ | ✓ |
| OPTICS | ✗ | ✗ | ✓ | Produces Ordering | ✓ |
| HDBSCAN | ✗ | ✗ | ✓ | ✓ | ✓ |